# PyTorch — Sorting, Selection & Autograd

This notebook covers two core PyTorch skills:
1. **Sorting & selecting values** in tensors (`sort`, `argsort`, `topk`, `kthvalue`)
2. **Automatic differentiation** with `requires_grad` and `.backward()` — the engine behind training neural networks

## Setup

In [ ]:
import torch

## 1. Sorting a 1D Tensor

`tensor.sort()` returns a **named tuple** `(values, indices)` — the sorted values, and the original positions they came from. By default it sorts in **ascending** order.

In [ ]:
tensor_array = torch.tensor([1, 5, 3])
tensor_array

In [ ]:
# Ascending sort (default)
tensor_array.sort()

In [ ]:
# Descending sort
tensor_array.sort(descending=True)

## 2. Sorting a 2D Tensor — the `dim` argument

For multi-dimensional tensors, you must tell PyTorch **which axis** to sort along using `dim`:
- `dim=0` → sorts **down each column** (compares values across rows)
- `dim=1` → sorts **across each row** (compares values across columns)

If you don't pass `dim`, PyTorch defaults to sorting along the **last dimension** (equivalent to `dim=1` for a 2D tensor — i.e., row-wise).

In [ ]:
tensor_array_2D = torch.tensor([[1, 5, 3],
                                 [21, 16, 9]])
tensor_array_2D

In [ ]:
# No dim specified -> sorts along the last dimension (row-wise here)
tensor_array_2D.sort()

In [ ]:
# dim=0 -> sort column-wise (each column sorted top-to-bottom)
tensor_array_2D.sort(dim=0)

In [ ]:
# dim=1 -> sort row-wise (each row sorted left-to-right)
tensor_array_2D.sort(dim=1)

## 3. `argsort` — Just the Indices

`torch.argsort(x)` gives you **only the indices** that would sort the tensor, without the sorted values themselves. Useful when you need to reorder *other* related data (e.g., labels) to match the sorted order.

In [ ]:
x = torch.tensor([3, 2, 6, 4, 9, 7])
indices = torch.argsort(x)
indices

## 4. `topk` — Top-K Largest Values

`torch.topk(x, k=3)` returns the **k largest values** and their indices, already sorted in descending order. This is commonly used to get, e.g., the top-3 predicted classes from a model's output logits.

In [ ]:
values, indices = torch.topk(x, k=3)
values, indices

In [ ]:
values

## 5. `kthvalue` — Find the k-th Smallest Value

`torch.kthvalue(x, k=1)` finds the **k-th smallest** value (i.e., searches from the bottom). Unlike `topk`, it only returns **one** value/index at a time — the one at rank `k`.

In [ ]:
values, indices = torch.kthvalue(x, k=1)  # k=1 -> the smallest value
values

---
## Gradient & Derivatives (Autograd)

PyTorch can automatically compute derivatives for you. Set `requires_grad=True` on a tensor to tell PyTorch: *"track every operation done on this, so I can later ask for the gradient."*

Calling `.backward()` on the final output computes `d(output)/d(input)` for every tensor in the chain that had `requires_grad=True`, and stores the result in `.grad`.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
a

`a` is a **leaf tensor** (created directly by us, not from an operation). Calling `.backward()` directly on it is the trivial case: `d(a)/d(a) = 1`.

In [ ]:
a.backward()
a.grad  # -> 1.0, since da/da = 1

### Building a computation graph

Now let's define `b = a + 2` and backward through it. Since `db/da = 1`, we'd expect `a.grad` to become `1.0` again.

⚠️ **Important gotcha:** PyTorch **accumulates** gradients into `.grad` by default — it *adds* the new gradient to whatever was already there, rather than overwriting it. That's why `a.grad` will show `2.0` below (the `1.0` from the previous cell + the new `1.0`), not `1.0`. In real training loops you call `optimizer.zero_grad()` (or `a.grad = None`) before each `.backward()` to avoid this.

In [ ]:
b = a + 2
b

In [ ]:
b.backward()

In [ ]:
a.grad  # accumulated: 1.0 (from a.backward()) + 1.0 (from b.backward()) = 2.0

Same accumulation happens again with `c = b + 5`. Since `dc/da = 1` as well, `a.grad` becomes `3.0` (2.0 + 1.0).

In [ ]:
c = b + 5
c

In [ ]:
c.backward()

In [ ]:
a.grad  # 3.0 total, due to accumulation across all three .backward() calls

### `grad_fn` — tracing how a tensor was created

Every non-leaf tensor (one created by an operation on a `requires_grad=True` tensor) has a `.grad_fn` attribute pointing to the function that produced it. This is how PyTorch builds the graph it walks backward through.

In [ ]:
c.grad_fn  # AddBackward0 -> c was created by an addition

### Try it yourself: avoiding accumulation
Reset the gradient and re-run to see the *actual* individual derivative, without the accumulation gotcha above.

In [ ]:
a2 = torch.tensor(2.0, requires_grad=True)
b2 = a2 + 2
c2 = b2 + 5
c2.backward()
a2.grad  # -> 1.0, the true dc2/da2 (no leftover accumulation this time)